## 🎯 Learning Objectives
* Deconstruct the core components of a GPT-like Transformer decoder architecture.
* Implement a masked Multi-Head Self-Attention mechanism from first principles in PyTorch.
* Construct a complete Transformer decoder block incorporating attention, feed-forward networks, residual connections, and layer normalization.
* Assemble these building blocks into a functional minimal GPT model capable of processing sequential token data.
* Gain practical experience with PyTorch's nn.Module API for complex model construction and forward pass definition.


## Exercise: Build a Minimal GPT in PyTorch

**Task:** Your goal is to implement a minimal Generative Pre-trained Transformer (GPT) model using PyTorch. This exercise will solidify your understanding of the core architectural components of modern large language models, specifically the decoder-only Transformer architecture.

**Requirements:**

1.  **`MultiHeadAttention` Module:** Implement a class for masked multi-head self-attention. This module should:
    *   Take query, key, and value tensors as input.
    *   Apply linear transformations to project Q, K, V for each head.
    *   Perform scaled dot-product attention.
    *   Apply a causal (look-ahead) mask to prevent attending to future tokens.
    *   Concatenate the outputs from all heads and apply a final linear projection.

2.  **`FeedForward` Module:** Implement a simple two-layer feed-forward network with a GELU activation and dropout.

3.  **`TransformerBlock` Module:** Implement a single decoder block. This block should:
    *   Consist of a `MultiHeadAttention` layer followed by `LayerNorm` and a residual connection.
    *   Followed by a `FeedForward` layer, another `LayerNorm`, and a residual connection.
    *   Incorporate dropout where appropriate.

4.  **`PositionalEncoding` Module:** Implement a sinusoidal positional encoding layer to inject positional information into the token embeddings.

5.  **`MinimalGPT` Module:** Assemble the above components into a full GPT model. This module should:
    *   Start with an `nn.Embedding` layer for token representations.
    *   Add `PositionalEncoding` to the token embeddings.
    *   Stack multiple `TransformerBlock`s.
    *   End with a final linear layer to project the output back to the vocabulary size (logits).

**Evaluation Criteria:**

*   **Correctness:** The forward pass of all modules should work without errors, and tensor shapes should be consistent throughout the model.
*   **Architectural Adherence:** The implementation must correctly reflect the decoder-only Transformer architecture, including masked self-attention and the order of operations within a Transformer block (attention -> add&norm -> feedforward -> add&norm).
*   **Code Quality:** Code should be clean, readable, well-commented, and follow PyTorch best practices (e.g., using `nn.Module`, `nn.Parameter`, `torch.Tensor` operations).
*   **Modularity:** Each component should be encapsulated in its own `nn.Module` class.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- Device Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters for our Minimal GPT ---
vocab_size = 10000  # Example vocabulary size
max_seq_len = 512   # Maximum sequence length the model can handle
batch_size = 4      # Example batch size
d_model = 768       # Dimension of the model's hidden states (embedding dimension)
n_heads = 12        # Number of attention heads
n_layers = 6        # Number of Transformer blocks
d_ff = d_model * 4  # Dimension of the feed-forward network (usually 4x d_model)
dropout_rate = 0.1  # Dropout rate

# --- Helper Function: Causal Mask Generation ---
def generate_causal_mask(seq_len):
    """
    Generates a square upper-triangular mask for causal attention.
    This mask prevents attention to future tokens.
    """
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask.to(device)

# --- Mock Data Generation ---
# Generate a dummy batch of input token IDs
dummy_input_tokens = torch.randint(0, vocab_size, (batch_size, max_seq_len)).to(device)

print(f"\nDummy input tokens shape: {dummy_input_tokens.shape}")
print(f"Causal mask shape (for seq_len={max_seq_len}): {generate_causal_mask(max_seq_len).shape}")


Now, it's your turn! Implement the `MultiHeadAttention`, `FeedForward`, `PositionalEncoding`, `TransformerBlock`, and `MinimalGPT` classes below. Use the hyperparameters defined in the setup cell.

Remember to pay close attention to tensor shapes, residual connections, layer normalization, and the application of the causal mask in your attention mechanism.


In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Implements masked Multi-Head Self-Attention.
    """
    def __init__(self, d_model, n_heads, dropout_rate=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # Linear layers for Q, K, V projections for all heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Output linear layer
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, query, key, value, mask=None):
        batch_size, seq_len, _ = query.shape

        # 1. Project Q, K, V and split into multiple heads
        # Shape: (batch_size, seq_len, d_model) -> (batch_size, seq_len, n_heads, head_dim)
        # Then permute to (batch_size, n_heads, seq_len, head_dim) for batch matrix multiplication
        Q = self.q_proj(query).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(key).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(value).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # 2. Calculate attention scores (Q @ K^T)
        # (batch_size, n_heads, seq_len, head_dim) @ (batch_size, n_heads, head_dim, seq_len)
        # -> (batch_size, n_heads, seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 3. Apply mask (for causal attention in GPT)
        if mask is not None:
            # Ensure mask is broadcastable: (1, 1, seq_len, seq_len) or (batch_size, 1, seq_len, seq_len)
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        # 4. Apply softmax to get attention probabilities
        attention_probs = F.softmax(attention_scores, dim=-1)
        attention_probs = self.dropout(attention_probs)

        # 5. Multiply with V to get weighted sum
        # (batch_size, n_heads, seq_len, seq_len) @ (batch_size, n_heads, seq_len, head_dim)
        # -> (batch_size, n_heads, seq_len, head_dim)
        context_layer = torch.matmul(attention_probs, V)

        # 6. Concatenate heads and apply final linear projection
        # (batch_size, n_heads, seq_len, head_dim) -> (batch_size, seq_len, n_heads, head_dim)
        # -> (batch_size, seq_len, d_model)
        context_layer = context_layer.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.out_proj(context_layer)

        return output


class FeedForward(nn.Module):
    """
    A simple two-layer feed-forward network with GELU activation.
    """
    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.gelu = nn.GELU() # Modern activation function
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class PositionalEncoding(nn.Module):
    """
    Injects sinusoidal positional information into the token embeddings.
    """
    def __init__(self, d_model, max_seq_len=512):
        super().__init__()
        self.d_model = d_model

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Add batch dimension: (1, max_seq_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x has shape (batch_size, seq_len, d_model)
        # Add positional encoding to the input embeddings
        # Ensure that the positional encoding is sliced to the current sequence length
        return x + self.pe[:, :x.size(1)].detach()


class TransformerBlock(nn.Module):
    """
    A single decoder block of the GPT architecture.
    Consists of masked multi-head attention, layer norm, residual connection,
    feed-forward network, layer norm, and another residual connection.
    """
    def __init__(self, d_model, n_heads, d_ff, dropout_rate=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout_rate)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout_rate)

        self.feed_forward = FeedForward(d_model, d_ff, dropout_rate)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):
        # Self-attention block
        # Apply LayerNorm BEFORE attention (Pre-LN Transformer, common in GPT-2/3)
        norm_x = self.norm1(x)
        attn_output = self.attention(norm_x, norm_x, norm_x, mask=mask)
        x = x + self.dropout1(attn_output) # Add residual connection

        # Feed-forward block
        norm_x = self.norm2(x)
        ff_output = self.feed_forward(norm_x)
        x = x + self.dropout2(ff_output) # Add residual connection

        return x


class MinimalGPT(nn.Module):
    """
    A minimal Generative Pre-trained Transformer model.
    """
    def __init__(self, vocab_size, max_seq_len, d_model, n_heads, n_layers, d_ff, dropout_rate=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)

        # Stack multiple Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout_rate)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model) # Final LayerNorm after all blocks
        self.output_head = nn.Linear(d_model, vocab_size) # Projects to vocabulary size for logits

        self.dropout = nn.Dropout(dropout_rate)

        # Initialize weights (optional, but good practice for Transformers)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids):
        # input_ids: (batch_size, seq_len)
        seq_len = input_ids.size(1)

        # 1. Token Embeddings
        x = self.token_embedding(input_ids) # (batch_size, seq_len, d_model)

        # 2. Positional Encoding
        x = self.positional_encoding(x) # (batch_size, seq_len, d_model)
        x = self.dropout(x)

        # 3. Generate causal mask
        # The mask needs to be (1, 1, seq_len, seq_len) for broadcasting with attention scores
        causal_mask = generate_causal_mask(seq_len).unsqueeze(0).unsqueeze(0)

        # 4. Pass through Transformer blocks
        for block in self.transformer_blocks:
            x = block(x, mask=causal_mask)

        # 5. Final LayerNorm and output projection
        x = self.final_norm(x)
        logits = self.output_head(x) # (batch_size, seq_len, vocab_size)

        return logits


# --- Instantiate and Test the MinimalGPT Model ---
print("\n--- Instantiating MinimalGPT ---")
model = MinimalGPT(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    dropout_rate=dropout_rate
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f} M")

# Perform a forward pass with dummy data
print(f"\nInput shape: {dummy_input_tokens.shape}")
output_logits = model(dummy_input_tokens)

print(f"Output logits shape: {output_logits.shape}")

# Verify output values (should not be NaNs or Infs)
assert not torch.isnan(output_logits).any(), "Output contains NaNs!"
assert not torch.isinf(output_logits).any(), "Output contains Infs!"
print("Forward pass successful and output is clean!")

# Example: Get the predicted token for the last position in the first sequence
predicted_token_id = torch.argmax(output_logits[0, -1, :]).item()
print(f"Predicted token ID for the last position of the first sequence: {predicted_token_id}")
